# RUN__pdf_ocr_summary — trạng thái parse báo cáo tài chính, theo cổ phiếu

Đọc **toàn bộ** `raw_data/cafef/financials/statements/**/*.csv` (mọi template), đối chiếu với
**chỉ mục PDF** qua chính `FinancialsBuilder.documents()`, trả về mỗi cổ phiếu một dòng, 6 cột.
⚠️ Chỉ đọc — không parse, không OCR, không ghi gì vào `raw_data/`.

| cột | nghĩa |
|---|---|
| `exchange` | sàn niêm yết, đọc từ chính cột `exchange` của CSV |
| `complete` | `bool` — **cả ba mốc `<= first_report`**, tức đoạn đã OCR của cả ba statement lùi về ít nhất tới chỗ chuỗi filing bắt đầu |
| `first_report` | quý **mở đầu chuỗi filing liền mạch** tới hiện tại, đọc từ `raw_data/cafef/pdfs/files/` — một sự thật về **filing**, không phải về parse |
| `balance_sheet` · `income_statement` · `cash_flow` | quý **mở đầu chuỗi liền mạch đã OCR được**, đi ngược từ quý đã đọc mới nhất theo **lịch quý** — một quý không có dòng `pdf` là đứt, kể cả quý không ai nộp |

Quý in dạng `2008-Q4` — sắp xếp được, và đúng dạng `--quarters` / `QUARTERS` của `pdf_ocr_job`,
nên một ô dán thẳng vào lệnh parse được. (CSV trên đĩa vẫn ghi `Q4-2008`.)

⚠️ **`first_report` ĐỌC TỪ FILE, KHÔNG TỪ CHỈ MỤC VÀ KHÔNG TỪ CSV.** Chỉ mục là thứ CafeF quảng
cáo, **file là thứ OCR mở được**. Hai nguồn không bằng nhau: đo 2026-08-31, đúng **1 quý trên 7 mã**
có filing trong chỉ mục mà **không có file PDF trên đĩa** — ACB 2009-Q3, và nó vẫn đang mang dòng
`pdf` đọc từ chính cái file đã biến mất. Cell 6 in cảnh báo đó.

⚠️ **`complete` ĐO ĐẦU CHUỖI, KHÔNG ĐO ĐỈNH.** Một quý còn thiếu ở **trên cùng** không kéo nó xuống:
BID đọc `True` trong khi 2026-Q2 chưa parse. Đoạn đã đọc chạm tới đâu là câu ba cột statement trả
lời, và `outstanding` ở cell 10 đếm chính xác còn bao nhiêu ô.

⚠️ **`—` ở một cột statement nghĩa là chưa có dòng `pdf` nào**, còn `—` ở `first_report` nghĩa là
không đọc được file PDF nào cho mã đó.

⚠️ **Bảng chỉ nói "đã xong từ đâu", KHÔNG nói quý nào còn thiếu.** Lưới `outstanding` ở cell 8 là
chỗ lấy danh sách đó ra.

⚠️ **Bằng chứng "quý này có filing" phải đến từ ngoài statements CSV.** Dòng `missing` không mang
`document`, nên chỉ đọc CSV thì *"không nộp"* và *"có filing mà parse hỏng"* là một chữ giống hệt
nhau. Ô còn thiếu có hai dạng và **cả hai đều tính**: `missing` (có dòng, đã thử, bị từ chối) và
`absent` (không có dòng nào cả).

⚠️ **Mẫu số ở đây là `allow_parent=True`, còn `pdf_ocr_job.plan()` mặc định `False`** — một quý
chỉ có báo cáo riêng lẻ vừa bị báo thiếu ở đây, vừa bị `plan()` trả lời *"files no document"*
trong khi file PDF nằm ngay trên đĩa. Đo 2026-08-30: ACB 4 quý, TCB 2, VIC 1, còn lại 0 — bật
`ALLOW_PARENT=True`.

⚠️ **`complete = False` là *đoạn đã OCR chưa lùi tới chỗ chuỗi filing bắt đầu*, không phải
*parser hỏng*.** Filing có thể không chứa statement đó (ACB 2009-Q2/Q3 là mẫu CBTT-03, không hề
có báo cáo lưu chuyển tiền tệ), hoặc là một KQKD lũy kế mà `pdf_ocr_merge` từ chối ghi (CTG 32 quý).

⚠️ **Chỉ mục PDF không nằm trong git** (`raw_data/` bị ignore trừ `financials/`), nên bản
checkout mới không có: ticker đó là `False` kèm cảnh báo ở cell 6, chứ không đoán (§5 rule 2).

Lịch sử ba lần sửa mốc (2026-08-30 / 08-31) và số đo của chúng: `kgpu/PDF_OCR.md` §8,
`CLAUDE.md` §6-2-sexquadragies.


## 1 · Định vị thư mục

⚠️ `CWD-1`: `fin.STATEMENTS_DIR` / `fin.PDFS_DIR` mặc định là đường dẫn **tương đối**, nên gọi từ
`src/kaggle_gpu/` sẽ đọc một thư mục rỗng — trông y hệt một ticker chưa parse bao giờ. Repo root
được dò ngược từ cwd, và cả hai thư mục dữ liệu lấy từ **một mỏ neo duy nhất**
(`pdf_ocr_job.use_data_root()`) rồi in ra.


In [16]:
import sys
from pathlib import Path

import pandas as pd

REPORTS = ["balance_sheet", "cash_flow", "income_statement"]
REL_STATEMENTS = Path("raw_data/cafef/financials/statements")


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / REL_STATEMENTS).is_dir():
            return candidate
    raise FileNotFoundError(f"khong tim thay {REL_STATEMENTS} tu {here} tro len")


REPO_ROOT = _repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from web_scraper import cafef_financials as fin  # noqa: E402
from web_scraper import pdf_ocr_job as job  # noqa: E402

# CWD-1: `fin.STATEMENTS_DIR` / `fin.PDFS_DIR` mac dinh la duong dan TUONG DOI, nen goi tu
# `src/kaggle_gpu/` se doc mot thu muc rong — trong y het mot ticker chua parse bao gio.
# `use_data_root` la duong CHINH THUC de tro lai (no set ca nam hang so cua module cung mot luc),
# nen o day khong co ban sao thu hai cua bat ky duong dan nao.
DATA_ROOT = job.use_data_root(REPO_ROOT / "raw_data" / "cafef")
STATEMENTS_DIR = Path(fin.STATEMENTS_DIR)
PDF_INDEX_DIR = Path(fin.PDFS_DIR) / "index"
PDF_FILES_DIR = Path(fin.PDFS_DIR) / "files"

print("cwd            :", Path.cwd())
print("repo root      :", REPO_ROOT)
print("data root      :", DATA_ROOT)
print("statements dir :", STATEMENTS_DIR)
print("pdf index dir  :", PDF_INDEX_DIR, "" if PDF_INDEX_DIR.is_dir() else "<- KHONG CO")
print("pdf files dir  :", PDF_FILES_DIR, "" if PDF_FILES_DIR.is_dir() else "<- KHONG CO")


cwd            : d:\GIT\master-thesis\src\kaggle_gpu
repo root      : D:\GIT\master-thesis
data root      : D:\GIT\master-thesis\raw_data\cafef
statements dir : D:\GIT\master-thesis\raw_data\cafef\financials\statements
pdf index dir  : D:\GIT\master-thesis\raw_data\cafef\pdfs\index 
pdf files dir  : D:\GIT\master-thesis\raw_data\cafef\pdfs\files 


## 2 · Đọc mọi CSV thành một bảng dài

`encoding='utf-8-sig'` là bắt buộc — cột đầu tiên của các file này mang BOM, nếu không sẽ thành
`﻿symbol`.

⚠️ `document` **không** được dùng để suy ra quý nào có filing: dòng `missing` không mang cột đó,
nên tên file chỉ nói được về những quý đã parse **thành công**. Bằng chứng filing đến từ cell 6.


In [17]:
META = ["symbol", "exchange", "template", "period", "year", "quarter", "source", "document"]

frames = []
for path in sorted(STATEMENTS_DIR.glob("*/*/*.csv")):
    report = path.parent.name
    if report not in REPORTS:
        print(f"WARNING: bo qua {path} — thu muc bao cao la {report!r}")
        continue
    part = pd.read_csv(path, usecols=META, encoding="utf-8-sig")
    part["report"] = report
    part["file"] = path.relative_to(REPO_ROOT).as_posix()
    frames.append(part)

if not frames:
    raise FileNotFoundError(f"khong co file .csv nao trong {STATEMENTS_DIR}")

records = pd.concat(frames, ignore_index=True)
records["source"] = records["source"].fillna("missing")


def quarter_rank(period: pd.Series) -> pd.Series:
    """`"Q3-2014"` -> 8059 — so nguyen tang dan, de lay min/max ma khong sap xep chuoi."""
    return period.str.slice(3).astype(int) * 4 + period.str.slice(1, 2).astype(int)


def quarter_id(period: pd.Series) -> pd.Series:
    """`"Q3-2014"` -> `"2014-Q3"` — sap xep duoc, va la dang `--quarters` cua `pdf_ocr_job`."""
    return period.str.slice(3) + "-" + period.str.slice(0, 2)


def rank_to_quarter(rank):
    """8059 -> `"2014-Q3"` — nghich dao cua `quarter_rank`, va TOAN PHAN.

    Mot bang tra `rank -> quarter_id` dung duoc cho nhung rank da co trong luoi va im lang
    tra `NaN` cho moi rank khac; `first_report` bay gio den tu THU MUC FILE, nen no co the
    la mot quy khong ticker nao trong luoi mang.
    """
    if pd.isna(rank):
        return None
    year, q = divmod(int(rank) - 1, 4)
    return f"{year}-Q{q + 1}"


# `period` la thu duy nhat CA HAI nguon (statements CSV va chi muc PDF) deu mang, nen moi phep so
# sanh quy o duoi deu di qua no — mot luat, hai bang. Cot `year`/`quarter` cua CSV phai dong y:
# neu khong thi mot trong hai da bi doc nham cot, va im lang o day se thanh mot phep join sai o
# cell 8, cho ma ket qua sai van trong hop ly.
disagree = records["period"] != ("Q" + records["quarter"].astype(int).astype(str)
                                 + "-" + records["year"].astype(int).astype(str))
if disagree.any():
    raise ValueError(f"{int(disagree.sum())} dong co `period` khong khop `year`/`quarter`")

# mot symbol niem yet tren hai san se pha khoa ticker — doi khoa thay vi im lang gop nham
pairs = records[["exchange", "symbol"]].drop_duplicates()
_PREFIX_EXCHANGE = bool(pairs["symbol"].duplicated().any())
TICKER_BY_PAIR = {(e, s): (f"{e}_{s}" if _PREFIX_EXCHANGE else s)
                  for e, s in pairs.itertuples(index=False)}


def ticker_key(exchange: pd.Series, symbol: pd.Series) -> pd.Series:
    """Khoa ticker — MOT luat, dung cho ca `records` lan `filings`."""
    return pd.Series([TICKER_BY_PAIR[k] for k in zip(exchange, symbol)], index=symbol.index)


records["ticker"] = ticker_key(records["exchange"], records["symbol"])
records["rank"] = quarter_rank(records["period"])
records["quarter_id"] = quarter_id(records["period"])

# `(ticker, report, period)` la khoa ma luoi o cell 8 join vao. Mot dong trung se nhan ban luoi
# do va lam moi phep dem o duoi sai theo mot huong khong ai nhin thay.
duplicated = int(records.duplicated(["ticker", "report", "period"]).sum())
if duplicated:
    raise ValueError(f"{duplicated} dong trung khoa (ticker, report, period)")

unexpected = sorted(set(records["source"]) - {"pdf", "missing"})
if unexpected:
    print(f"WARNING: gia tri source ngoai du kien: {unexpected} — "
          f"bang duoi coi chung KHONG phai 'co bao cao'")

print(f"{len(frames)} file / {records['ticker'].nunique()} co phieu / {len(records)} dong")
print(records["source"].value_counts().to_dict())
records


21 file / 7 co phieu / 1311 dong
{'pdf': 1174, 'missing': 137}


,symbol,exchange,template,period,year,quarter,source,document,report,file,ticker,rank,quarter_id
0,ACB,HOSE,bank,Q1-2008,2008,1,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8033,2008-Q1
1,ACB,HOSE,bank,Q2-2008,2008,2,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8034,2008-Q2
2,ACB,HOSE,bank,Q3-2008,2008,3,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8035,2008-Q3
3,ACB,HOSE,bank,Q4-2008,2008,4,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8036,2008-Q4
4,ACB,HOSE,bank,Q1-2009,2009,1,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8037,2009-Q1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1306,VIC,HOSE,corp,Q3-2024,2024,3,pdf,Q3-2024_bao_cao_tai_chinh_hop_nhat_quy_3_nam_2...,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8099,2024-Q3
1307,VIC,HOSE,corp,Q4-2024,2024,4,missing,NaN,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8100,2024-Q4
1308,VIC,HOSE,corp,Q1-2025,2025,1,missing,NaN,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8101,2025-Q1
1309,VIC,HOSE,corp,Q2-2025,2025,2,missing,NaN,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8102,2025-Q2


## 3 · Chỉ mục PDF — quý nào **thực sự** có filing

Một quý `missing` chỉ đáng gọi là *thiếu* khi doanh nghiệp **có nộp** báo cáo cho quý đó, và
statements CSV không trả lời được câu đó.

⚠️ **Gọi thẳng `FinancialsBuilder.documents()`, không chép lại luật chọn filing** (hợp nhất trước
rồi mới tới mức soát xét; báo cáo năm đã kiểm toán đứng thay Q4 nhưng không bao giờ đổi entity;
sàn `FINANCIALS_PERIOD_MIN = Q1-2008`) — bản sao thứ hai sẽ lệch ngay lần đầu bản gốc đổi.

⚠️ **`allow_parent=True` — rộng nhất trong những gì có trên đĩa.** Một quý chỉ có báo cáo riêng lẻ
vẫn là một filing **chưa parse**; đếm nó vào là bảo thủ đúng hướng.

⚠️ **`quarterly_filing` đọc thẳng cột `quarter` của chỉ mục, không đọc cờ `annual` của filing ĐƯỢC
CHỌN.** `documents()` trả về đúng một filing mỗi quý và ưu tiên bản năm đã kiểm toán ở Q4, nên cờ
đó trả lời *"filing được chọn có phải bản quý không"* — câu khác hẳn. ACB 2009-Q4 nộp cả hai bản
mà cột cũ vẫn `False`; sửa xong **98 quý đổi cờ trên 7 cổ phiếu, 0 mốc đổi chỗ**, và chiều ngược
được **assert** ở cell 6 (0 vi phạm). ⚠️ **Từ 2026-08-31 cột này không còn ai đọc** — mốc chuỗi
filing mà nó phục vụ đã bị bỏ cùng luật `complete` cũ; nó vẫn được tính và vẫn được assert.

⚠️ **Và `first_report` KHÔNG đọc từ chỉ mục — nó đọc từ `pdfs/files/`.** Chỉ mục nói CafeF có gì,
file nói ta có gì để OCR; hai cái lệch nhau ở đúng 1 quý trên 7 mã (ACB 2009-Q3, có trong chỉ mục
và không có trên đĩa) và cell 6 in cảnh báo. Tên file mang chính kỳ của nó (`Q3-2011_…`, `FY-2016_…`),
báo cáo **năm** được gộp vào Q4 đúng như `documents()` gộp, và 3 file `NA-<năm>` trên toàn bộ 784 mã
bị bỏ qua vì không đọc được quý (§5 rule 2).

⚠️ **`parent_only`** = quý mà bản duy nhất tồn tại là báo cáo **riêng lẻ**, tức quý `plan()` ở mặc
định `allow_parent=False` không nhìn thấy. Đo 2026-08-30: ACB 4 (đúng cả bốn quý ACB còn thiếu),
TCB 2, VIC 1, còn lại 0.


In [18]:
import csv  # noqa: E402
import re  # noqa: E402

builder = fin.FinancialsBuilder()


def quarterly_filed(exchange: str, symbol: str) -> set:
    """Nhung quy ma ticker nay CO NOP mot bao cao QUY — doc thang cot `quarter` cua chi muc.

    ⚠️ **KHONG DOC DUOC DIEU NAY TU `documents()`, VA DO LA LOI DA SUA 2026-08-30.**
    `documents()` tra ve DUNG MOT filing moi quy va uu tien ban bao cao NAM da kiem toan o
    Q4, nen co `annual` cua no tra loi *"filing DUOC CHON co phai bao cao quy khong"* — khong
    phai *"ticker co nop bao cao quy khong"*. Hai cau khac nhau, va **ACB 2009-Q4 la vi du
    ngay trong bo du lieu nay**: chi muc mang ca "Bao cao tai chinh quy 4 nam 2009" (quarter=4)
    lan ban nam da kiem toan (quarter=5), `documents()` chon ban nam, va cot cu tra loi
    `False` cho mot quy MA ACB CO nop bao cao quy. Do 2026-08-30 tren 7 ticker: **98 quy doi
    tu False sang True**, khong quy nao doi nguoc lai.

    ⚠️ **Khong mot `first_report` nao trong 7 ticker hom nay doi cho** — moi chuoi hien tai
    deu bat dau o mot quy khong phai Q4. Day la sua mot loi TIEM AN: ticker nao co chuoi bat
    dau dung o mot Q4 vua co bao cao quy vua co ban nam se bi day moc tre mot quy va **ba o
    lang le roi khoi mau so**, dung hinh dang ma notebook nay da phai sua hai lan (moc lay tu
    ket qua parse; mot filing le loi mo dau chuoi). MOC PHAI DOC TU SU KIEN CUA CHI MUC.

    ⚠️ **Khong loc `consolidated`:** mau so o cell nay la `allow_parent=True`, nen phep thu
    "co bao cao quy khong" phai rong dung bang the — loc hep hon se lam ACB 2009-Q4 (bao cao
    RIENG LE) rot lai. Day la mot CAU HOI KHAC chu khong phai ban sao cua luat chon filing
    (hop nhat truoc, muc soat xet sau); luat do van chi nam trong `documents()`.
    """
    with open(PDF_INDEX_DIR / f"{exchange}_{symbol}.csv", encoding="utf-8-sig") as f:
        return {f"Q{int(r['quarter'])}-{r['year']}" for r in csv.DictReader(f)
                if str(r["quarter"]).isdigit() and int(r["quarter"]) in (1, 2, 3, 4)}


rows, no_index = [], []
for listed_on, symbol in records[["exchange", "symbol"]].drop_duplicates().itertuples(index=False):
    try:
        docs = builder.documents(listed_on, symbol, allow_parent=True)
    except FileNotFoundError:
        no_index.append(f"{listed_on}_{symbol}")
        continue
    quarterly = quarterly_filed(listed_on, symbol)
    # Chieu nguoc lai phai luon dung: mot filing ma `documents()` goi la bao cao quy thi trong
    # chi muc BAT BUOC co mot dong quarter 1..4. Neu khong, hai ben dang doc chi muc theo hai
    # luat khac nhau va cot `quarterly_filing` se HEP hon su that ma khong ai thay.
    # (Do 2026-08-30: 0 vi pham tren ca 7 ticker.)
    stale = [d["period"] for d in docs
             if d["annual"] == "False" and d["period"] not in quarterly]
    if stale:
        raise ValueError(f"{listed_on}_{symbol}: {len(stale)} filing quy cua `documents()` "
                         f"khong co dong quarter 1..4 nao trong chi muc: {stale[:5]}")
    # ⚠️ CUNG MOT CAU HOI, O DO RONG MAC DINH CUA CONG CU PARSE. Chenh lech giua hai ben la
    # nhung quy CHI ton tai duoi dang bao cao RIENG LE, va do la cay cau giua hai notebook:
    # `pdf_ocr_job.plan()` o `allow_parent=False` bao "files no document for quarter(s) [...]"
    # trong khi file PDF nam ngay tren dia. ACB 2009-Q2/Q3/Q4 la dung ba quy do.
    consolidated_only = {d["period"] for d in
                         builder.documents(listed_on, symbol, allow_parent=False)}
    rows += [(symbol, listed_on, d["period"], d["period"] in quarterly,
              d["period"] not in consolidated_only) for d in docs]

if no_index:
    print("WARNING: khong co chi muc PDF cho:", ", ".join(no_index))
    print("         -> khong chung minh duoc quy nao co filing, nen cac ticker nay luon "
          "`complete=False` (§5 rule 2: thieu phep do thi la thieu, khong suy dien)")

filings = pd.DataFrame(rows, columns=["symbol", "exchange", "period", "quarterly_filing",
                                      "parent_only"])
if filings.empty:
    raise FileNotFoundError(f"khong doc duoc chi muc PDF nao trong {PDF_INDEX_DIR}")
filings["ticker"] = ticker_key(filings["exchange"], filings["symbol"])
filings["rank"] = quarter_rank(filings["period"])
filings["quarter_id"] = quarter_id(filings["period"])

# `documents()` tra ve dung mot filing moi quy (no gom theo `period`), va luoi o cell 8 dua vao
# dieu do — neu no doi, phep dem duoi kia phong len ma khong bao gi.
duplicated = int(filings.duplicated(["ticker", "period"]).sum())
if duplicated:
    raise ValueError(f"{duplicated} filing trung khoa (ticker, period)")

# Kiem tra nguoc, va no re: moi dong `pdf` phai roi vao mot quy CO filing trong chi muc. Neu khong
# thi hai nguon dang noi hai chuyen khac nhau va bang o cell 8 khong doc duoc.
filed = set(map(tuple, filings[["ticker", "period"]].itertuples(index=False)))
orphan = sorted({(t, p) for t, p, s in zip(records["ticker"], records["period"], records["source"])
                 if s == "pdf" and (t, p) not in filed})
if orphan:
    print(f"WARNING: {len(orphan)} quy co dong `pdf` ma chi muc khong co filing nao:", orphan[:10])

PDF_PERIOD = re.compile(r"^(?:Q([1-4])|FY)-(\d{4})_", re.IGNORECASE)


def quarters_on_disk(exchange: str, symbol: str) -> set:
    """{rank} cac quy CO FILE PDF that su nam tren dia, doc tu TEN FILE.

    ⚠️ Bao cao NAM duoc gop vao Q4 — dung phep gop ma `documents()` lam — nen hai ben dem
    cung mot don vi. 3 file tren toan bo 784 ma mang tien to `NA-<nam>` (CafeF khong ghi
    quy) va bi bo qua: khong doc duoc quy thi khong dem (§5 rule 2).
    """
    out = set()
    for path in (PDF_FILES_DIR / f"{exchange}_{symbol}").glob("*.pdf"):
        m = PDF_PERIOD.match(path.name)
        if m:
            out.add(int(m.group(2)) * 4 + (int(m.group(1)) if m.group(1) else 4))
    return out


def chain_start(ranks: set) -> int:
    """Quy mo dau chuoi lien mach, di NGUOC tu quy moi nhat co file.

    Cham vao mot quy khong co file nao la DUT: mot filing le loi nam truoc cho dut khong
    mo dau chuoi nao ca. Neo o file MOI NHAT CUA CHINH TICKER chu khong o quy lich hien
    tai — mot ma da huy niem yet van co chuoi cua rieng no.
    """
    ordered = sorted(ranks)
    start = ordered[-1]
    for r in reversed(ordered[:-1]):
        if r != start - 1:
            break
        start = r
    return start


# ⚠️ `first_report` DOC TU `pdfs/files/`, khong tu statements CSV va khong tu chi muc. Cau
# hoi la *"tu quy nao tro di ta CO filing lien mach toi hien tai"*, va bang chung cho no la
# FILE nam tren dia: chi muc chi la thu CafeF quang cao, con file la thu OCR mo duoc.
chain, no_files, gone = {}, [], []
for listed_on, symbol in filings[["exchange", "symbol"]].drop_duplicates().itertuples(
        index=False):
    ticker = TICKER_BY_PAIR[(listed_on, symbol)]
    ranks = quarters_on_disk(listed_on, symbol)
    if not ranks:
        no_files.append(f"{listed_on}_{symbol}")
        continue
    chain[ticker] = chain_start(ranks)
    gone += [(ticker, rank_to_quarter(r))
             for r in sorted(set(filings.loc[filings["ticker"] == ticker, "rank"]) - ranks)]

filing_chain = pd.Series(chain, dtype="float64", name="first_filing")

if no_files:
    print(f"WARNING: khong co file PDF nao trong {PDF_FILES_DIR} cho:", ", ".join(no_files))
if gone:
    # Mot quy co filing trong chi muc ma khong co file tren dia la quy KHONG parse lai duoc
    # — va no van co the dang mang dong `pdf` doc tu chinh file da bien mat.
    print(f"WARNING: {len(gone)} quy co filing trong chi muc ma KHONG co file PDF tren dia "
          f"(khong parse lai duoc):", gone[:10])

# ⚠️ SETTLED — mot statement ma mot lan chay TRUOC DAY da CHUNG MINH la filing khong he
# chua. Doc tu chinh `job.settled_absences`, khong chep lai luat: mot bao cao chi la SETTLED
# khi MOI ly do duoc ghi lai deu la `no such statement on any page of this filing` — mot
# layer tim thay trang roi tu choi vi so hoc thi day la loi parse, con thang duoc.
#
# ⚠️ **VA DAY LA LY DO NO CO MAT TRONG BANG NAY (sua 2026-09-02).** Truoc do mot o nhu the
# lam DUT chuoi o cell 8 y het mot o parse hong, nen `complete` cua nhung ma do khong bao gio
# co the `True` — khong phai vi parser hong ma vi doanh nghiep khong nop bao cao do. TCB co
# BON bao cao luu chuyen tien te nhu vay (Q4-2009, Q4-2010, Q1-2017, Q3-2017) va CTG mot.
# §5 rule 24: quy nao khong co PDF doc duoc thi `missing` LA cau tra loi dung.
#
# ⚠️ Chi nhung o DA DUOC DO. Mot o chua ai do van lam dut chuoi, dung nhu the (§5 rule 2), va
# mot run folder cu hon artefact schema v4 khong ghi ly do nen khong dong gop gi.
SETTLED_CELLS = set()
for listed_on, symbol in filings[["exchange", "symbol"]].drop_duplicates().itertuples(
        index=False):
    ticker = TICKER_BY_PAIR[(listed_on, symbol)]
    for quarter, reports in job.settled_absences(
            REPO_ROOT / "reports" / "pdf_ocr", listed_on, symbol).items():
        for report in reports:
            SETTLED_CELLS.add((ticker, quarter, report))

if SETTLED_CELLS:
    print(f"{len(SETTLED_CELLS)} o da duoc DO la filing khong he chua statement do "
          f"(SETTLED, khong lam dut chuoi):")
    for key in sorted(SETTLED_CELLS):
        print("   ", " ".join(key))

print(f"{filings['ticker'].nunique()} co phieu / {len(filings)} filing / "
      f"{int(filings['quarterly_filing'].sum())} filing quy / "
      f"{int(filings['parent_only'].sum())} quy CHI co bao cao rieng le "
      f"(can ALLOW_PARENT=True moi mo duoc)")
filings


8 o da duoc DO la filing khong he chua statement do (SETTLED, khong lam dut chuoi):
    ACB 2008-Q1 balance_sheet
    ACB 2008-Q1 cash_flow
    ACB 2009-Q2 cash_flow
    TCB 2009-Q4 cash_flow
    TCB 2010-Q4 cash_flow
    TCB 2017-Q1 cash_flow
    TCB 2017-Q3 cash_flow
    VIC 2008-Q2 income_statement
7 co phieu / 419 filing / 405 filing quy / 7 quy CHI co bao cao rieng le (can ALLOW_PARENT=True moi mo duoc)


,symbol,exchange,period,quarterly_filing,parent_only,ticker,rank,quarter_id
0,ACB,HOSE,Q1-2008,True,True,ACB,8033,2008-Q1
1,ACB,HOSE,Q2-2009,True,True,ACB,8038,2009-Q2
2,ACB,HOSE,Q3-2009,True,True,ACB,8039,2009-Q3
3,ACB,HOSE,Q4-2009,True,True,ACB,8040,2009-Q4
4,ACB,HOSE,Q1-2010,True,False,ACB,8041,2010-Q1
...,...,...,...,...,...,...,...,...
414,VIC,HOSE,Q1-2025,True,False,VIC,8101,2025-Q1
415,VIC,HOSE,Q2-2025,True,False,VIC,8102,2025-Q2
416,VIC,HOSE,Q3-2025,True,False,VIC,8103,2025-Q3
417,VIC,HOSE,Q4-2025,True,False,VIC,8104,2025-Q4


## 4 · Bảng tóm tắt — 6 cột

Ba cột statement là **quý mở đầu chuỗi liền mạch đã OCR được**, đi ngược theo **lịch quý** từ
quý **mới nhất đã đọc** của chính statement đó — cùng luật `chain_start` mà `first_report` dùng,
nên một lỗ ở trên cùng không làm rỗng cột.
`first_report` là **quý mở đầu chuỗi filing liền mạch**, đọc từ `pdfs/files/` ở cell 6: nó nói ta
**có gì để đọc**, còn ba cột kia nói ta **đã đọc tới đâu**. `complete` là `bool` (để lọc thẳng bằng
`summary[summary["complete"]]`): **cả ba mốc `<= first_report`**, tức cả ba đã lùi về ít nhất tới
chỗ chuỗi filing bắt đầu. Một cột `—` là NaN, mọi so sánh với NaN trả `False`, nên thiếu một
statement là hàng đó tự rơi về `False`.

Lưới kỳ vọng là `mọi quý có filing × ba statement`; một ô chưa có dòng `pdf` có hai dạng:

| dạng | nghĩa | ví dụ |
|---|---|---|
| `missing` | có dòng, đã thử, bị một gate từ chối hoặc `pdf_ocr_merge` không ghi | ACB lưu chuyển tiền tệ **2009-Q2/Q3** |
| `absent` | **không có dòng nào** — lưới CSV chưa với tới quý đó | VIC 45 quý sau 2014-Q4; BID **2026-Q2** |

⚠️ **Quý không ai nộp LÀM ĐỨT chuỗi ở ba cột statement**, đúng như nó làm đứt chuỗi filing ở
`first_report`: cả hai đều liền mạch theo **lịch quý**. BSR lưu chuyển tiền tệ đọc `2018-Q2` vì
Q1-2018 không có filing nào; ba quý đã parse trước đó (Q4-2016, Q3-2017, Q4-2017) nằm **bên kia**
chỗ đứt. Mẫu số của `outstanding` ở cell 10 thì vẫn chỉ là những quý CÓ filing.

⚠️ **`complete` ĐO ĐẦU CHUỖI, KHÔNG ĐO ĐỈNH — và đó là chỗ duy nhất nó không nhìn thấy.** Một quý
còn thiếu ở **trên cùng** không kéo nó xuống: BID đọc `True` trong khi 2026-Q2 chưa parse. Ba cột
statement là chỗ đọc "đã tới đâu", và `outstanding` ở cell 10 đếm chính xác còn bao nhiêu ô.

⚠️ **`complete = False` là *đoạn đã OCR chưa lùi tới chỗ chuỗi filing bắt đầu*, không phải *parser
hỏng*.** Filing có thể không chứa statement đó (§5 rule 24), hoặc là một KQKD lũy kế không có
Q1..Q(q-1) để trừ (CTG 32 quý). Mã không có chỉ mục PDF, hoặc không có file PDF nào, cũng là
`False` — không đoán (§5 rule 2).


In [19]:
exchange_of = records.groupby("ticker")["exchange"].first().rename("exchange")
index = pd.Index(sorted(records["ticker"].unique()), name="ticker")

# LUOI KY VONG: moi quy CO FILING x ba statement. Mot o chua co dong `pdf` la mot o CON THIEU, va
# `missing` (co dong, da thu, bi tu choi) lan `absent` (khong co dong nao) deu tinh — dem mot loai
# ma bo loai kia chinh la cach mot lan chay bi dung giua chung trong nhu da xong (VIC, 45 quy).
expected = filings[["ticker", "period", "rank", "quarter_id", "parent_only"]].merge(
    pd.DataFrame({"report": REPORTS}), how="cross")
expected = expected.merge(records[["ticker", "period", "report", "source"]],
                          how="left", on=["ticker", "period", "report"])
expected["state"] = expected["source"].fillna("absent")
# ⚠️ MOT O SETTLED KHONG LAM DUT CHUOI. No la mot su that ve FILING (filing khong chua
# statement do), khong phai ve parser — va coi no nhu mot o parse hong khien `complete` cua
# TCB va CTG khong bao gio co the `True`, du khong con gi de OCR. No VAN nam trong
# `outstanding` o cell 10, va cot `settled` o do dem rieng chung ra.
expected["settled"] = [k in SETTLED_CELLS for k in
                       zip(expected["ticker"], expected["quarter_id"], expected["report"])]

outstanding = expected[expected["state"] != "pdf"]


# BA COT STATEMENT: quy MO DAU chuoi lien mach da OCR duoc, di nguoc tu quy da doc moi nhat.


def read_from(group: pd.DataFrame):
    """Quy mo dau chuoi lien mach cac quy da co dong `pdf`, di NGUOC tu quy da doc moi nhat.

    ⚠️ LIEN MACH THEO LICH QUY — dung `chain_start`, chinh luat ma `first_report` dung, nen
    hai con so tren cung mot hang doc duoc voi nhau. Mot rank khong mang dong `pdf` la DUT,
    du ly do la parse hong (`missing`/`absent`) hay la KHONG AI NOP quy do. BSR luu chuyen
    tien te doc `2018-Q2`: Q1-2018 khong co filing nao, nen chuoi dung o day va ba quy da
    parse truoc do (Q4-2016, Q3-2017, Q4-2017) nam BEN KIA cho dut, khong noi dai chuoi.

    ⚠️ Neo o quy MOI NHAT DA DOC cua chinh statement do, nen mot lo hong o TREN CUNG khong
    lam rong ca cot: BID thieu dung 2026-Q2 va ba cot cua no van noi OCR da lui toi dau. Doi
    lai, cot nay mot minh KHONG cho biet doan da doc co cham toi filing moi nhat hay khong —
    `first_report` va `complete` tra loi cau do.
    """
    read = set(group.loc[group["state"] == "pdf", "rank"])
    if not read:
        return None
    # ⚠️ BAC QUA cac o SETTLED, va CHI nhung o nam duoi quy da doc moi nhat: neo van la mot
    # quy THAT SU DA DOC, nen mot o SETTLED o tren cung khong tu no keo moc len.
    top = max(read)
    bridged = {r for r in group.loc[group["settled"], "rank"] if r <= top}
    return chain_start(read | bridged)


read_rank = (expected.groupby(["ticker", "report"])[["rank", "state", "settled"]]
             .apply(read_from).unstack("report").reindex(index=index, columns=REPORTS))
read_quarter = read_rank.apply(lambda col: col.map(rank_to_quarter))

# `first_report` = quy MO DAU chuoi filing lien mach toi hien tai, doc tu `pdfs/files/` o
# cell 6. Day la mot su that ve FILING, khong phai ve parse: no noi ta co gi de doc, con ba
# cot tren noi ta da doc toi dau.
first_rank = filing_chain.reindex(index)
first_report = first_rank.map(rank_to_quarter).rename("first_report")

# `complete` = ca ba moc deu `<= first_report`, tuc doan da OCR cua ca ba statement lui ve toi
# it nhat cho chuoi filing bat dau. Mot cot `—` la NaN, va moi phep so sanh voi NaN tra ve
# False, nen ticker nao chua doc duoc mot statement nao do — hoac khong co file PDF nao — tu
# roi ve False, khong can chan rieng.
#
# ⚠️ NO DO DAU CHUOI, KHONG DO DINH CHUOI. Mot quy con thieu o TREN CUNG khong keo `complete`
# xuong: BID doc `True` trong khi 2026-Q2 chua parse. Doan da doc cham toi dau la cau hoi ba
# cot statement tra loi, va cot `outstanding` o cell 10 dem chinh xac con bao nhieu o.
complete = read_rank.le(first_rank, axis=0).all(axis=1).rename("complete")

summary = pd.DataFrame(index=index).join(exchange_of).join(first_report).join(read_quarter)
text_cols = ["first_report", *REPORTS]
summary[text_cols] = summary[text_cols].fillna("—")
# `complete` join SAU fillna, de no giu kieu bool thay vi bi doi thanh chuoi
summary = summary.join(complete)[["exchange", "complete", *text_cols]]

summary

,exchange,complete,first_report,balance_sheet,cash_flow,income_statement
ticker,,,,,,
ACB,HOSE,True,2009-Q4,2009-Q2,2009-Q4,2009-Q2
BID,HOSE,True,2011-Q3,2011-Q3,2011-Q3,2011-Q3
BSR,HOSE,True,2018-Q2,2018-Q2,2018-Q2,2018-Q2
CTG,HOSE,False,2008-Q4,2008-Q4,2008-Q4,2015-Q1
TCB,HOSE,False,2012-Q2,2013-Q2,2021-Q2,2012-Q4
VCB,HOSE,True,2008-Q4,2008-Q4,2008-Q4,2008-Q4
VIC,HOSE,False,2008-Q2,2024-Q3,2025-Q3,2024-Q1


## 5 · Phụ — đếm quý đã parse / còn thiếu

Mẫu số để đọc bảng trên: `first_report` sớm mà `coverage` thấp nghĩa là chuỗi dài nhưng rỗng.

⚠️ **`coverage` chia cho `filed`** — số quý **thực sự có filing** — **không chia cho số dòng
CSV**. `rows` phồng lên vì quý không ai nộp, và hụt đi vì quý có filing mà lưới CSV chưa với tới
(`outstanding` dạng `absent`). Chia cho `rows` là trộn lịch nộp báo cáo vào phép đo parser.


In [20]:
tally = (
    records.groupby(["ticker", "report"])["source"]
    .value_counts()
    .unstack("source")
    .reindex(columns=["pdf", "missing"])
    .fillna(0)
    .astype(int)
)
tally["rows"] = tally["pdf"] + tally["missing"]
tally["filed"] = tally.index.get_level_values("ticker").map(filings.groupby("ticker").size())
tally["outstanding"] = (outstanding.groupby(["ticker", "report"]).size()
                        .reindex(tally.index).fillna(0).astype(int))
# ⚠️ `settled` la phan cua `outstanding` DA DUOC DO la khong the parse — filing khong chua
# statement do — nen `outstanding - settled` moi la so o mot lan chay lai con co the thang.
tally["settled"] = (outstanding[outstanding["settled"]].groupby(["ticker", "report"]).size()
                    .reindex(tally.index).fillna(0).astype(int))
tally["winnable"] = tally["outstanding"] - tally["settled"]
tally["coverage"] = (tally["pdf"] / tally["filed"]).round(3)
tally


source                   pdf  missing  rows  filed  outstanding  settled  \
ticker report                                                              
ACB    balance_sheet      68        5    73     69            1        1   
       cash_flow          66        7    73     69            3        2   
       income_statement   68        5    73     69            1        0   
BID    balance_sheet      62        8    70     63            1        0   
       cash_flow          61        9    70     63            2        0   
       income_statement   59       11    70     63            4        0   
BSR    balance_sheet      12        5    17     14            2        0   
       cash_flow          14        3    17     14            0        0   
       income_statement   14        3    17     14            0        0   
CTG    balance_sheet      70        0    70     70            0        0   
       cash_flow          70        0    70     70            0        0   
       income_statement   69        1    70     70            1        0   
TCB    balance_sheet      57       10    67     61            4        0   
       cash_flow          52       15    67     61            9        4   
       income_statement   59        8    67     61            2        0   
VCB    balance_sheet      70        0    70     70            0        0   
       cash_flow          70        0    70     70            0        0   
       income_statement   70        0    70     70            0        0   
VIC    balance_sheet      60       10    70     72           12        0   
       cash_flow          58       12    70     72           14        0   
       income_statement   45       25    70     72           27        1   

source                   winnable  coverage  
ticker report                                
ACB    balance_sheet            0     0.986  
       cash_flow                1     0.957  
       income_statement         1     0.986  
BID    balance_sheet            1     0.984  
       cash_flow                2     0.968  
       income_statement         4     0.937  
BSR    balance_sheet            2     0.857  
       cash_flow                0     1.000  
       income_statement         0     1.000  
CTG    balance_sheet            0     1.000  
       cash_flow                0     1.000  
       income_statement         1     0.986  
TCB    balance_sheet            4     0.934  
       cash_flow                5     0.852  
       income_statement         2     0.967  
VCB    balance_sheet            0     1.000  
       cash_flow                0     1.000  
       income_statement         0     1.000  
VIC    balance_sheet           12     0.833  
       cash_flow               14     0.806  
       income_statement        26     0.625